In [5]:
%pip install numpy pandas duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 54.9 MB/s  0:00:006m0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import duckdb

conn = duckdb.connect("main.db")

In [10]:
# Which pollutants appear most frequently as "prominent pollutants"?

conn.execute(""" 
WITH pollutants_unnested AS (
SELECT
    date,
    state,
    area,
    air_quality_status,
    aqi_value,
    UNNEST(STRING_SPLIT(prominent_pollutants, ',')) AS pollutants        
FROM air_quality
)

/*
SELECT
    pollutants,
    AVG(aqi_value) AS avg_aqi,
    COUNT(DISTINCT date) AS pollutant_occurence
FROM pollutants_unnested
GROUP BY pollutants
ORDER BY pollutant_occurence DESC
*/
             
SELECT
    pollutants,
    COUNT(DISTINCT date) AS total_occurrences,
    AVG(aqi_value) AS avg_aqi_when_present,
    COUNT(DISTINCT area) AS cities_affected
FROM pollutants_unnested
WHERE air_quality_status IN ('Poor', 'Very Poor', 'Severe')
GROUP BY pollutants
ORDER BY total_occurrences DESC
""").fetch_df()

,pollutants,total_occurrences,avg_aqi_when_present,cities_affected
0,PM2.5,2938,278.565462,261
1,PM10,2660,267.305885,210
2,O3,1532,244.716811,184
3,NO2,309,254.996904,56
4,CO,18,289.277778,13
5,SO2,13,250.153846,5


In [27]:
# Do multiple pollutants appear together?
conn.execute("""

WITH pollutant_combos AS (
    SELECT
        date,
        area,
        prominent_pollutants,
        LENGTH(prominent_pollutants) - LENGTH(REPLACE(prominent_pollutants, ',', '')) + 1 AS num_pollutants
    FROM air_quality
    WHERE air_quality_status IN ('Poor', 'Very Poor', 'Severe')
)

SELECT
    prominent_pollutants,
    COUNT(*) AS occurrences,
    AVG(num_pollutants) AS avg_pollutants_together
FROM pollutant_combos
GROUP BY prominent_pollutants
ORDER BY occurrences DESC
LIMIT 20
""").fetch_df()

,prominent_pollutants,occurrences,avg_pollutants_together
0,PM2.5,45091,1.0
1,PM10,10258,1.0
2,"PM2.5,PM10",3979,2.0
3,O3,1808,1.0
4,"PM2.5,O3",425,2.0
5,NO2,214,1.0
6,"PM10,O3",168,2.0
7,"O3,PM2.5,PM10",126,3.0
8,"PM2.5,NO2",62,2.0
9,"PM10,NO2",25,2.0


*PM2.5 is the most commonly occurring pollutant and since it is also one of the deadliest pollutants - the company needs to prioritise PM2.5 targeting*

In [30]:
# Which cities consistently show Poor/Very Poor/Severe AQI?

conn.execute(""" 

WITH city_stats AS (
    SELECT
        area,
        state,
        COUNT(DISTINCT date) AS total_days_monitored,
        COUNT(DISTINCT CASE 
            WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') 
            THEN date END) AS bad_air_days,
        AVG(CASE 
            WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') 
            THEN aqi_value END) AS avg_bad_day_aqi,
        MAX(aqi_value) AS worst_aqi_ever
    FROM air_quality
    GROUP BY area, state
),

bad_air_days_pct AS (
SELECT
    area,
    state,
    total_days_monitored,
    bad_air_days,
    ROUND(100.0 * bad_air_days / total_days_monitored, 2) AS pct_bad_days,
    avg_bad_day_aqi,
    worst_aqi_ever
FROM city_stats
WHERE bad_air_days > 100  -- Filter for cities with sustained issues
ORDER BY pct_bad_days DESC, bad_air_days DESC
)
             
SELECT
    CASE WHEN pct_bad_days > 60.0 THEN 'Crisis'
    WHEN pct_bad_days BETWEEN 40.0 AND 60.0 THEN 'High Risk'
    WHEN pct_bad_days BETWEEN 20.0 AND 39.9 THEN 'Mid Risk'
    ELSE 'Low Risk'
    END AS risk_tier,
    COUNT(area) AS cities
FROM bad_air_days_pct
GROUP BY 1
""").fetch_df()

,risk_tier,cities
0,Crisis,1
1,High Risk,16
2,Mid Risk,60
3,Low Risk,62


*Cities with more than 50% bad days => desperate need => strong market fit*

In [16]:
# What's the trend over time - improving or degrading?

conn.execute(""" 
WITH yearly_avg AS (
    SELECT
        area,
        EXTRACT('year' FROM date) AS year,
        AVG(aqi_value) AS avg_aqi,
        COUNT(DISTINCT CASE 
            WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') 
            THEN date END) AS bad_days,
        COUNT(DISTINCT date) AS total_days
    FROM air_quality
    GROUP BY area, EXTRACT('year' FROM date)
),
trend_calc AS (
    SELECT
        area,
        year,
        avg_aqi,
        bad_days,
        total_days,
        LAG(avg_aqi) OVER (PARTITION BY area ORDER BY year) AS prev_year_aqi,
        LAG(bad_days) OVER (PARTITION BY area ORDER BY year) AS prev_year_bad_days
    FROM yearly_avg
)

SELECT
    area,
    year,
    avg_aqi,
    prev_year_aqi,
    ROUND(avg_aqi - prev_year_aqi, 2) AS aqi_change,
    bad_days,
    prev_year_bad_days,
    bad_days - prev_year_bad_days AS bad_days_change,
    CASE 
        WHEN avg_aqi - prev_year_aqi > 10 THEN 'Worsening'
        WHEN avg_aqi - prev_year_aqi < -10 THEN 'Improving'
        ELSE 'Stable'
    END AS trend_status
FROM trend_calc
ORDER BY area, year

""").fetch_df()

,area,year,avg_aqi,prev_year_aqi,aqi_change,bad_days,prev_year_bad_days,bad_days_change,trend_status
0,Agartala,2020,153.450980,NaN,NaN,8,<NA>,<NA>,Stable
1,Agartala,2021,102.346505,153.450980,-51.10,44,8,36,Improving
2,Agartala,2022,111.895028,102.346505,9.55,86,44,42,Stable
3,Agartala,2023,155.321637,111.895028,43.43,108,86,22,Worsening
4,Agartala,2024,139.898413,155.321637,-15.42,94,108,-14,Improving
...,...,...,...,...,...,...,...,...,...
1626,Yamunanagar,2020,154.491124,174.106628,-19.62,95,114,-19,Improving
1627,Yamunanagar,2021,176.817647,154.491124,22.33,122,95,27,Worsening
1628,Yamunanagar,2022,165.410828,176.817647,-11.41,100,122,-22,Improving
1629,Yamunanagar,2023,132.850794,165.410828,-32.56,41,100,-59,Improving


In [ ]:
# Cities with worsening trends
conn.execute("""
WITH yearly_avg AS (
    SELECT
        area,
        EXTRACT('year' FROM date) AS year,
        AVG(aqi_value) AS avg_aqi
    FROM air_quality
    GROUP BY area, EXTRACT('year' FROM date)
),

year_bounds AS (
    SELECT
        area,
        MIN(year) AS first_year,
        MAX(year) AS last_year
    FROM yearly_avg
    GROUP BY area
)

SELECT
    yb.area,
    yb.first_year,
    yb.last_year,
    y1.avg_aqi AS early_aqi,
    y2.avg_aqi AS recent_aqi,
    y2.avg_aqi - y1.avg_aqi AS aqi_change
FROM year_bounds yb
JOIN yearly_avg y1 
    ON yb.area = y1.area AND yb.first_year = y1.year
JOIN yearly_avg y2 
    ON yb.area = y2.area AND yb.last_year = y2.year
WHERE y2.avg_aqi > y1.avg_aqi + 20   -- significant worsening
ORDER BY aqi_change DESC
""").fetch_df()

,area,first_year,last_year,early_aqi,recent_aqi,aqi_change
0,Darbhanga,2021,2023,248.863636,364.714286,115.850649
1,Hajipur,2020,2025,98.784553,196.636364,97.851811
2,Panchkula,2015,2025,92.000000,185.416667,93.416667
3,Gurgaon,2015,2018,146.333333,214.957198,68.623865
4,Tumakuru,2023,2025,95.818182,151.854545,56.036364
5,Gangtok,2022,2025,33.594444,82.962963,49.368519
6,Kunjemura,2023,2025,89.469613,137.254777,47.785164
7,Milupara,2023,2025,60.752874,105.792079,45.039206
8,Shillong,2019,2025,35.615385,75.927928,40.312543
9,Tumidih,2023,2025,81.409756,120.717172,39.307416


In [31]:
# Combine AQI + Population for market potential

conn.execute("""
WITH city_pollution AS (
    SELECT
        area,
        state,
        AVG(aqi_value) AS avg_aqi,
        COUNT(DISTINCT CASE 
            WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') 
            THEN date END) * 100.0 / COUNT(DISTINCT date) AS pct_bad_days,
        COUNT(DISTINCT date) AS total_days
    FROM air_quality
    GROUP BY area, state
),
state_population AS (
    SELECT
        state,
        AVG(CASE WHEN gender = 'Total' THEN value * 1000 END) AS population
    FROM population_data
    WHERE year = 2024  -- or most recent year available
    GROUP BY state
)

SELECT
    c.area,
    c.state,
    c.avg_aqi,
    c.pct_bad_days,
    c.total_days,
    p.population,
    CASE 
        WHEN c.pct_bad_days > 60 THEN 'Tier 1 Priority'
        WHEN c.pct_bad_days > 40 THEN 'Tier 2 Priority'
        ELSE 'Lower Priority'
    END AS market_tier
FROM city_pollution c
LEFT JOIN state_population p ON c.state = p.state
WHERE c.total_days > 1000  -- At least ~3 years of data
AND p.population > 2000000  -- At least 2M people (market size threshold)
ORDER BY c.pct_bad_days DESC
LIMIT 20
""").fetch_df()

,area,state,avg_aqi,pct_bad_days,total_days,population,market_tier
0,Delhi,Delhi,215.895080,51.372656,3679,2.180300e+07,Tier 2 Priority
1,Bhiwadi,Rajasthan,206.061728,49.421296,2592,2.201900e+07,Tier 2 Priority
2,Ghaziabad,Uttar Pradesh,213.321598,49.322952,2954,5.782467e+07,Tier 2 Priority
3,Greater Noida,Uttar Pradesh,206.425068,47.216816,2569,5.782467e+07,Tier 2 Priority
4,Faridabad,Haryana,200.388872,44.894939,3379,1.315067e+07,Tier 2 Priority
5,NOIDA,Uttar Pradesh,199.969409,43.881713,2942,5.782467e+07,Tier 2 Priority
6,Gurugram,Haryana,188.784468,42.400332,2408,1.315067e+07,Tier 2 Priority
7,Patna,Bihar,186.660480,41.856000,3125,1.599267e+07,Tier 2 Priority
8,Muzaffarpur,Bihar,184.824534,41.764148,3163,1.599267e+07,Tier 2 Priority
9,Chhapra,Bihar,186.193727,40.498155,1084,1.599267e+07,Tier 2 Priority


In [ ]:
# Seasonal patterns in AQI

conn.execute(""" 
WITH seasonal_data AS (
    SELECT
        area,
        state,
        date,
        CASE WHEN EXTRACT('month' FROM date) IN (11,12,1,2) THEN 'Winter'
             WHEN EXTRACT('month' FROM date) IN (3,4,5,6) THEN 'Summer'
             WHEN EXTRACT('month' FROM date) IN (7,8,9,10) THEN 'Monsoon'
        END AS season,
        air_quality_status
    FROM air_quality
),
city_seasonal_stats AS (
    SELECT
        area,
        season,
        COUNT(*) AS total_days,
        COUNT(CASE WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') THEN 1 END) AS bad_days,
        COUNT(CASE WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') THEN 1 END) * 100.0 / COUNT(*) AS pct_bad_days
    FROM seasonal_data
    GROUP BY area, season
),
city_overall_risk AS (
    SELECT
        area,
        AVG(CASE 
             WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') 
             THEN 1 ELSE 0 END) * 100 AS overall_pct_bad
    FROM air_quality
    GROUP BY area
)

SELECT
    CASE 
        WHEN c.overall_pct_bad >= 40 THEN 'High Risk'
        WHEN c.overall_pct_bad >= 20 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS city_tier,
    s.season,
    COUNT(DISTINCT s.area) AS num_cities,
    ROUND(AVG(s.pct_bad_days), 1) AS avg_pct_bad_days
FROM city_seasonal_stats s
JOIN city_overall_risk c ON s.area = c.area
GROUP BY city_tier, s.season
ORDER BY city_tier, season
""").fetch_df()

,city_tier,season,num_cities,avg_pct_bad_days
0,High Risk,Monsoon,19,21.5
1,High Risk,Summer,18,41.0
2,High Risk,Winter,18,81.1
3,Low Risk,Monsoon,212,1.4
4,Low Risk,Summer,217,2.8
5,Low Risk,Winter,213,12.2
6,Medium Risk,Monsoon,59,7.9
7,Medium Risk,Summer,59,18.1
8,Medium Risk,Winter,59,55.3


High Risk Cities (18-19 cities):

- Winter: 4 out of 5 days are hazardous
- Summer: 2 out of 5 days still bad
- Monsoon: 1 out of 5 days bad

**Since we do not have city population data - I'm using the census2011 data from Kaggle - https://www.kaggle.com/datasets/faisaljanjua0555/top-500-biggest-cities-of-india**

In [3]:

conn.execute("CREATE TABLE IF NOT EXISTS indian_cities AS " \
            "SELECT * FROM '/workspaces/Product-Market-Fit-Analysis---Case-Study/Dataset/indian_cities.csv'")

conn.execute("SELECT * FROM indian_cities LIMIT 5").fetchdf()

,Rank,City,State,Population,Metro_Population,Sexratio,Literacy
0,1,Mumbai,Maharashtra,12442373,18414288,853,89.73
1,2,Delhi,Delhi,11034555,16314838,876,87.59
2,3,Bangalore,Karnataka,8443675,8499399,923,88.71
3,4,Hyderabad,Andhra Pradesh,6731790,7749334,955,83.26
4,5,Ahmedabad,Gujarat,5577940,6352254,898,88.29


In [5]:
conn.execute("""
-- Step 1: Calculate state-level growth rate from your projection data
WITH state_growth AS (
    SELECT
        state,
        AVG(CASE WHEN year = 2011 THEN value * 1000 END) AS pop_2011,
        AVG(CASE WHEN year = 2025 THEN value * 1000 END) AS pop_2025,
        (AVG(CASE WHEN year = 2025 THEN value * 1000 END) / 
         AVG(CASE WHEN year = 2011 THEN value * 1000 END)) AS growth_multiplier
    FROM population_data
    WHERE gender = 'Total'
    GROUP BY state
)

SELECT * FROM state_growth
ORDER BY growth_multiplier DESC
""").fetch_df()

,state,pop_2011,pop_2025,growth_multiplier
0,Dadra and Nagar Haveli,1.666667e+05,5.840000e+05,3.504000
1,Daman and Diu,1.893333e+05,6.513333e+05,3.440141
2,Sikkim,1.580000e+05,3.786667e+05,2.396624
3,Nagaland,5.810000e+05,1.120000e+06,1.927711
4,Tripura,9.763333e+05,1.769333e+06,1.812223
5,Kerala,1.621433e+07,2.889333e+07,1.781962
6,Ladakh,6.266667e+04,9.900000e+04,1.579787
7,Haryana,8.935667e+06,1.348767e+07,1.509419
8,Chhattisgarh,5.992000e+06,8.627667e+06,1.439864
9,Uttarakhand,3.076333e+06,4.403333e+06,1.431358


In [13]:
# Which Cities to Prioritize

conn.execute(""" 
WITH city_risk AS (
    SELECT
        state,
        area,
        COUNT(*) AS total_days,
        COUNT(CASE WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') THEN 1 END) * 100.0 / COUNT(*) AS pct_bad_days,
        AVG(aqi_value) AS avg_aqi,
        MAX(aqi_value) AS worst_aqi
    FROM air_quality
    GROUP BY state, area
    HAVING pct_bad_days >= 40
)

SELECT
    c.state,
    c.area,
    ROUND(c.pct_bad_days, 1) AS pct_bad_days,
    ROUND(c.avg_aqi, 0) AS avg_aqi,
    c.worst_aqi,
    c.total_days
FROM city_risk c
WHERE c.total_days >= 1000  -- More than 3 years of monitoring
""").fetch_df()

,state,area,pct_bad_days,avg_aqi,worst_aqi,total_days
0,Bihar,Chhapra,40.5,186.0,452.0,1084
1,Uttar Pradesh,Greater Noida,47.2,206.0,500.0,2569
2,Delhi,Delhi,51.4,216.0,497.0,3679
3,Haryana,Faridabad,44.9,200.0,496.0,3379
4,Madhya Pradesh,Singrauli,40.1,177.0,426.0,2471
5,Bihar,Patna,41.9,187.0,488.0,3125
6,Rajasthan,Bhiwadi,49.4,206.0,483.0,2592
7,Uttar Pradesh,NOIDA,43.9,200.0,500.0,2942
8,Bihar,Muzaffarpur,41.8,185.0,500.0,3163
9,Uttar Pradesh,Ghaziabad,49.3,213.0,500.0,2954


In [11]:
conn.execute("""
WITH state_growth AS (
    SELECT
        state,
        AVG(CASE WHEN year = 2011 AND gender = 'Total' THEN value * 1000 END) AS pop_2011,
        AVG(CASE WHEN year = 2025 AND gender = 'Total' THEN value * 1000 END) AS pop_2025,
        (AVG(CASE WHEN year = 2025 AND gender = 'Total' THEN value * 1000 END) / 
         AVG(CASE WHEN year = 2011 AND gender = 'Total' THEN value * 1000 END)) AS growth_multiplier
    FROM population_data
    GROUP BY state
),
census_cities AS (
    SELECT
        City,
        State,
        Metro_Population AS pop_2011,
        CASE 
            WHEN City = 'Delhi' THEN 'Delhi'
            WHEN State = 'Uttar Pradesh' THEN 'Uttar Pradesh'
            WHEN State = 'Haryana' THEN 'Haryana'
            WHEN State = 'Bihar' THEN 'Bihar'
            WHEN State = 'Rajasthan' THEN 'Rajasthan'
            WHEN State = 'Madhya Pradesh' THEN 'Madhya Pradesh'
            ELSE State
        END AS state_normalized
    FROM indian_cities
    WHERE City IN ('Ghaziabad', 'Noida', 'Delhi', 'Faridabad', 
                   'Gurgaon', 'Patna', 'Muzaffarpur')
       OR City LIKE '%Greater Noida%'
       OR City LIKE '%Baghpat%'
       OR City LIKE '%Bhiwadi%'
       OR City LIKE '%Singrauli%'
       OR City LIKE '%Chhapra%'
)

SELECT
    c.City AS area,
    c.State,
    c.pop_2011,
    ROUND(g.growth_multiplier, 3) AS growth_rate,
    ROUND((c.pop_2011 * g.growth_multiplier)/1000000, 1) AS pop_2025_estimate_in_Mil
FROM census_cities c
LEFT JOIN state_growth g ON c.state_normalized = g.state
ORDER BY pop_2025_estimate_in_Mil DESC
""").fetch_df()

,area,State,pop_2011,growth_rate,pop_2025_estimate_in_Mil
0,Delhi,Delhi,16314838,1.347,22.0
1,Ghaziabad,Uttar Pradesh,2358525,1.311,3.1
2,Patna,Bihar,2046652,1.377,2.8
3,Faridabad,Haryana,1414050,1.509,2.1
4,Gurgaon,Haryana,901968,1.509,1.4
5,Noida,Uttar Pradesh,637272,1.311,0.8
6,Muzaffarpur,Bihar,393724,1.377,0.5
7,Singrauli,Madhya Pradesh,220257,1.288,0.3
8,Bhiwadi,Rajasthan,104921,1.303,0.1
9,Greater Noida,Uttar Pradesh,102054,1.311,0.1


In [26]:
conn.execute("""
WITH state_growth AS (
    SELECT
        state,
        AVG(CASE WHEN year = 2011 AND gender = 'Total' THEN value * 1000 END) AS pop_2011,
        AVG(CASE WHEN year = 2025 AND gender = 'Total' THEN value * 1000 END) AS pop_2025,
        (AVG(CASE WHEN year = 2025 AND gender = 'Total' THEN value * 1000 END) / 
         AVG(CASE WHEN year = 2011 AND gender = 'Total' THEN value * 1000 END)) AS growth_multiplier
    FROM population_data
    GROUP BY state
),
census_cities AS (
    SELECT
        CASE WHEN City = 'Gurgaon' THEN 'Gurugram' ELSE City END AS City,
        CASE 
            WHEN City = 'Delhi' THEN 'Delhi'
            WHEN State = 'Uttar Pradesh' THEN 'Uttar Pradesh'
            WHEN State = 'Haryana' THEN 'Haryana'
            WHEN State = 'Bihar' THEN 'Bihar'
            WHEN State = 'Rajasthan' THEN 'Rajasthan'
            WHEN State = 'Madhya Pradesh' THEN 'Madhya Pradesh'
            ELSE State
        END AS state_normalized,
        Metro_Population AS metro_pop_2011
    FROM indian_cities
    WHERE City IN ('Ghaziabad', 'Noida', 'Delhi', 'Faridabad', 
                   'Gurgaon', 'Patna', 'Muzaffarpur')
       OR City LIKE '%Greater Noida%'
       OR City LIKE '%Baghpat%'
       OR City LIKE '%Bhiwadi%'
       OR City LIKE '%Singrauli%'
       OR City LIKE '%Chhapra%'
),
city_population AS (
    SELECT
        c.City AS area,
        c.state_normalized AS state,
        ROUND(c.metro_pop_2011 * g.growth_multiplier, 0) AS pop_2025_estimate
    FROM census_cities c
    LEFT JOIN state_growth g ON c.state_normalized = g.state
),
city_risk AS (
    SELECT
        state,
        area,
        COUNT(*) AS total_days,
        COUNT(CASE WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') THEN 1 END) * 100.0 / COUNT(*) AS pct_bad_days,
        AVG(aqi_value) AS avg_aqi,
        MAX(aqi_value) AS worst_aqi
    FROM air_quality
    GROUP BY state, area
    HAVING pct_bad_days >= 40
)

SELECT
    c.state,
    c.area,
    ROUND(c.pct_bad_days, 1) AS pct_bad_days,
    ROUND(c.avg_aqi, 0) AS avg_aqi,
    c.worst_aqi,
    c.total_days,
    ROUND(COALESCE(p.pop_2025_estimate, 0) / 1000000, 2) AS city_pop_millions,
    -- Updated scoring with CITY population
    ROUND(c.pct_bad_days * 0.4 + (c.avg_aqi / 3) * 0.4 + (COALESCE(p.pop_2025_estimate, 0) / 1000000) * 0.2, 1) AS priority_score
FROM city_risk c
LEFT JOIN city_population p ON LOWER(c.area) = LOWER(p.area) AND LOWER(c.state) = LOWER(p.state)
WHERE c.total_days >= 1000
ORDER BY priority_score DESC
LIMIT 20
""").fetch_df()

,state,area,pct_bad_days,avg_aqi,worst_aqi,total_days,city_pop_millions,priority_score
0,Delhi,Delhi,51.4,216.0,497.0,3679,21.97,53.7
1,Uttar Pradesh,Ghaziabad,49.3,213.0,500.0,2954,3.09,48.8
2,Rajasthan,Bhiwadi,49.4,206.0,483.0,2592,0.14,47.3
3,Uttar Pradesh,Greater Noida,47.2,206.0,500.0,2569,0.13,46.4
4,Haryana,Faridabad,44.9,200.0,496.0,3379,2.13,45.1
5,Uttar Pradesh,NOIDA,43.9,200.0,500.0,2942,0.84,44.4
6,Haryana,Gurugram,42.4,189.0,486.0,2408,1.36,42.4
7,Bihar,Patna,41.9,187.0,488.0,3125,2.82,42.2
8,Bihar,Muzaffarpur,41.8,185.0,500.0,3163,0.54,41.5
9,Bihar,Chhapra,40.5,186.0,452.0,1084,0.00,41.0


### **Top 5 Launch Cities:**

**Phase 1A (Year 1, Q4 2025):**
1. **Delhi** - 22M pop, 51% bad days → Flagship market
2. **Ghaziabad** - 3.1M pop, 49% bad days → NCR core
3. **Faridabad** - 2.1M pop, 45% bad days → NCR south

**Phase 1B (Year 2):**
4. **Noida** - High risk, NCR connectivity
5. **Gurugram** - Corporate hub

**Phase 2 (Year 2-3):**
- Bihar cluster: Patna (2.8M), Muzaffarpur
- Standalone: Bhiwadi (industrial)

In [2]:
conn.execute(""" 
-- AQI volatility
WITH daily_city_aqi AS (
    SELECT
        area,
        date,
        aqi_value,
        LAG(aqi_value) OVER (PARTITION BY area ORDER BY date) AS prev_day_aqi
    FROM air_quality
    -- only checking in priority cities
    WHERE area IN ('Delhi', 'Ghaziabad', 'Faridabad', 'Noida', 'Gurugram')
)

SELECT
    area,
    AVG(ABS(aqi_value - prev_day_aqi)) AS avg_daily_change,
    MAX(ABS(aqi_value - prev_day_aqi)) AS max_daily_swing,
    STDDEV(aqi_value) AS aqi_volatility
FROM daily_city_aqi
WHERE prev_day_aqi IS NOT NULL
GROUP BY area
ORDER BY avg_daily_change DESC

""").fetch_df()

,area,avg_daily_change,max_daily_swing,aqi_volatility
0,Faridabad,42.209591,446.0,106.299490
1,Ghaziabad,39.540129,293.0,110.341376
2,Gurugram,39.341919,296.0,91.398436
3,Delhi,34.317020,255.0,102.772086


In [3]:
conn.execute(""" 
SELECT
    area,
    COUNT(CASE WHEN air_quality_status = 'Severe' THEN 1 END) AS severe_days,
    ROUND(COUNT(CASE WHEN air_quality_status = 'Severe' THEN 1 END) * 100.0 / COUNT(*), 1) AS pct_severe
FROM air_quality
WHERE area IN ('Delhi', 'Ghaziabad', 'Faridabad', 'Noida', 'Gurugram')
GROUP BY area
ORDER BY pct_severe DESC
""").fetch_df()

,area,severe_days,pct_severe
0,Ghaziabad,188,6.4
1,Faridabad,165,4.9
2,Delhi,164,4.5
3,Gurugram,34,1.4


*Output from Perplexity*

Most air purifiers sold in India share a small set of core features; below is a simplified estimate of what each does and what it roughly adds to BOM/manufacturing cost for a mid‑range unit (not MRP).

### Common features, use case, and rough added cost

| Feature | Use case | Approx. added cost to include (India, mid‑range BOM) |
| --- | --- | --- |
| Pre‑filter (washable mesh/foam) | Captures large dust, hair and lint so that the HEPA and carbon filters last longer and clog less often.  [smarthousegears](https://smarthousegears.com/articles/best-air-purifiers-for-indian-homes) | ₹200–₹400 for basic mesh/foam frame and mounting.  [estimatorflorida](https://estimatorflorida.com/air-filters-cost-estimator/) |
| HEPA filter (H13 “True HEPA”) | Removes fine particulate matter (PM2.5, pollen, smoke, pet dander), typically 99.95% efficiency at 0.3 µm, which is critical for Indian city smog.  [smarthousegears](https://smarthousegears.com/articles/best-air-purifiers-for-indian-homes) | Retail replacement filters run about ₹1,500–₹3,000 for home purifiers; in OEM quantity, the incremental manufacturing cost is roughly ₹800–₹1,500 per unit.  [latestcost](https://latestcost.com/cost-hepa-filter-price-u-s-buyers/) |
| Activated carbon filter | Adsorbs gases and odours from traffic, cooking, VOCs and smoke; usually used with HEPA in Indian metros.  [smarthousegears](https://smarthousegears.com/articles/best-air-purifiers-for-indian-homes) | Small carbon modules retail around ₹800–₹1,500; at scale, media + frame typically adds about ₹400–₹900 per unit.  [aajjo](https://www.aajjo.com/product/activated-carbon-filter-in-bengaluru-urban-usmoon-water-india-private-limited) |
| UV‑C light (germicidal LED/tube) | Inactivates bacteria, viruses and mold spores passing through the purifier; marketed for health and allergy concerns.  [smarthousegears](https://smarthousegears.com/articles/best-air-purifiers-for-indian-homes) | UV‑C LEDs or a compact lamp with basic driver adds roughly ₹300–₹700 to parts cost, depending on power and design.  [estimatorflorida](https://estimatorflorida.com/air-filters-cost-estimator/) |
| Ionizer / plasma generator | Charges particles so they clump and get trapped more easily, sometimes marketed for killing microbes or removing smoke; often an optional toggle.  [smarthousegears](https://smarthousegears.com/articles/best-air-purifiers-for-indian-homes) | Simple ionizer assemblies for consumer devices typically add about ₹200–₹500 to BOM.  [indiamart](https://www.indiamart.com/proddetail/intelligent-air-purifier-with-carbon-hepa-filter-ionizer-uv-1928879155.html) |
| Air quality sensor (PM2.5 + VOC) | Measures particulate levels and sometimes gases so the purifier can show real‑time AQI and auto‑adjust fan speed.  [smarthousegears](https://smarthousegears.com/articles/best-air-purifiers-for-indian-homes) | Low‑cost PM2.5 + VOC sensor modules in volume are usually in the ₹500–₹1,200 range including basic housing and wiring.  [estimatorflorida](https://estimatorflorida.com/air-filters-cost-estimator/) |
| CADR‑matched high‑flow fan and housing | Provides enough airflow (CADR 300–400+ m³/h) for Indian bedroom/living‑room sizes while keeping noise acceptable.  [smarthousegears](https://smarthousegears.com/articles/best-air-purifiers-for-indian-homes) | Up‑sizing from a low‑end fan to a stronger, quieter unit plus more robust housing typically adds around ₹800–₹1,500 over a very basic design.  [smarthousegears](https://smarthousegears.com/articles/best-air-purifiers-for-indian-homes) |
| Auto mode + multiple fan speeds | Lets the purifier ramp up/down automatically based on sensor input, and gives manual control over noise vs performance.  [smarthousegears](https://smarthousegears.com/articles/best-air-purifiers-for-indian-homes) | Extra cost is mainly electronics and firmware: roughly ₹200–₹400 for a more capable controller and buttons/indicators over a 1–2‑speed basic design.  [estimatorflorida](https://estimatorflorida.com/air-filters-cost-estimator/) |
| Sleep mode & timer | Reduces fan speed and display brightness at night and lets user schedule shut‑off, improving comfort and power use.  [smarthousegears](https://smarthousegears.com/articles/best-air-purifiers-for-indian-homes) | Implemented in the same controller as auto mode; marginal extra BOM is small, about ₹50–₹150 for additional indicators/remote logic.  [reddit](https://www.reddit.com/r/HonestBuyerReviews/comments/1kz2qx7/top_10_best_air_purifiers_in_india_2025_tested/) |
| Digital display / indicator lights (PM2.5, AQI, filter change) | Shows air quality level (numbers or color ring) and filter‑replacement alerts, improving usability and perceived quality.  [smarthousegears](https://smarthousegears.com/articles/best-air-purifiers-for-indian-homes) | Segment/LED display plus drivers typically adds about ₹200–₹600 over a bare‑bones LED‑only design.  [smarthousegears](https://smarthousegears.com/articles/best-air-purifiers-for-indian-homes) |
| Smart app / Wi‑Fi connectivity | Remote control, scheduling, AQ history and integration with smart‑home ecosystems; common in mid/high‑end Indian models.  [smarthousegears](https://smarthousegears.com/articles/best-air-purifiers-for-indian-homes) | Wi‑Fi MCU module and extra certification typically add around ₹400–₹900 to BOM.  [estimatorflorida](https://estimatorflorida.com/air-filters-cost-estimator/) |
| Integrated humidifier (in some Indian models) | Adds moisture in dry seasons while purifying, aimed at comfort and throat/skin dryness issues.  [smarthousegears](https://smarthousegears.com/articles/best-air-purifiers-for-indian-homes) | Small evaporative/ultrasonic module, tank, and controls usually add ₹700–₹1,500 in parts.  [smarthousegears](https://smarthousegears.com/articles/best-air-purifiers-for-indian-homes) |

These costs are indicative per‑unit additions for a consumer purifier made at scale in India; retail prices will be higher because they include overheads, margins, distribution and tax.

## 🎯 Feature Prioritization Framework

### **Must-Have (Core Product)**

Based on your data:

| Feature | Why (Data-Driven) | Cost |
|---------|-------------------|------|
| **True HEPA (H13)** | PM2.5 = 73% of bad days; also catches PM10 | ₹800-1,500 |
| **Pre-filter** | Extends HEPA life in dusty Indian conditions | ₹200-400 |
| **High CADR fan (300-400 m³/h)** | Winter AQI avg 161 needs powerful filtration | ₹800-1,500 |
| **PM2.5 sensor + auto mode** | Need real-time response (run volatility query to confirm) | ₹700-1,600 |

**Total core BOM:** ~₹2,500-5,000

---

### **Should-Have (Competitive Parity)**

| Feature | Why | Cost |
|---------|-----|------|
| **Activated carbon filter** | O3 (3% of bad days) + odor complaints in cities | ₹400-900 |
| **Multi-speed + sleep mode** | 81% bad winter days = needs 24/7 runtime | ₹250-550 |
| **Digital display (AQI/PM2.5)** | Customers need to see it's working | ₹200-600 |
| **Filter change indicator** | User retention, recurring revenue | Minimal |

**Incremental:** ~₹850-2,050

---

### **Differentiation Options (Pick 1-2)**

| Feature | Competitive Advantage | Cost | Risk |
|---------|----------------------|------|------|
| **Smart app + Wi-Fi** | Corporate/tech-savvy NCR market | ₹400-900 | Development time |
| **Seasonal presets** | "Winter Warrior Mode" based on your seasonal data | ₹50-150 | Just firmware |
| **UV-C** | Health positioning (limited data to support) | ₹300-700 | Ozone concerns |


---

## 🎯 Recommended MVP

**Launch Product BOM: ₹3,350-7,550**
- True HEPA H13 + Pre-filter + Carbon
- High CADR fan
- PM2.5 sensor + auto mode
- Multi-speed + sleep mode  
- Digital AQI display
- **Differentiator:** Seasonal mode presets (low-cost firmware feature tied to data insight)

**Retail target:** ₹15,000-20,000 (3-4x BOM is typical)
